# Track 4: "The Assistant That Doesn’t Make Things Up"
## RNAP Hack 2026 — Fase 1: Dimensión 3 (Línea Base)
### Sistema de Recuperación Legal y Prevención de Alucinaciones con Citación Exacta

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

Este cuaderno implementa de principio a fin el pipeline de la **Fase 1 (Línea Base)** para responder preguntas ciudadanas sobre normatividad pública (**Reglamento de Tránsito y Vialidad de Apodaca, N.L.**), fundamentando cada respuesta en el artículo exacto y **rechazando estrictamente** preguntas que se encuentren fuera del marco normativo.

---
## Reporte Técnico — Sección 1: Problema y Datos

### 1.1 Contexto y Desafío del Track 4
En la asistencia jurídica y regulatoria ciudadana, las **alucinaciones** de los modelos de lenguaje representan un riesgo crítico: inventar normativas inexistentes, atribuir sanciones erróneas o fundamentar respuestas en artículos equivocados vulnera la certeza jurídica y la confianza del usuario.

El **Track 4 ("The Assistant That Doesn’t Make Things Up")** aborda este problema mediante una arquitectura de recuperación controlada que persigue dos objetivos ineludibles:
1. **Fidelidad y Fundamentación**: Para toda consulta legítima contemplada en la norma, el sistema debe recuperar el fragmento normativo exacto (artículo canónico) dentro de sus primeras posiciones (**Top-3**).
2. **Abstención Rigurosa (Anti-Alucinación)**: Si la consulta no está cubierta por el reglamento (preguntas capciosas, fueras de dominio o temas no legislados localmente), el sistema debe **rechazar explícitamente** la consulta en lugar de generar una respuesta plausible pero falsa.

### 1.2 Fuente de Datos Elegida
Se seleccionó como corpus de referencia el **Reglamento de Tránsito y Vialidad del Municipio de Apodaca, Nuevo León** (aplicable a nivel metropolitano). 
* **Estructura Documental**: Disposiciones generales, facultades de autoridades, derechos peatonales, obligaciones de conductores, límites de velocidad, zonas escolares, prohibiciones (uso de celular, alcoholímetro), requerimientos para ciclistas/motociclistas, siniestros y régimen de sanciones.
* **Granularidad de Segmentación**: En el ámbito legal, la unidad lógica y jurídica elemental es el **Artículo**. Por tanto, la fragmentación (*chunking*) se realiza a nivel de artículo completo, garantizando que el contexto normativo conserve su integridad semántica y número identificador unívoco.

### 1.3 Tamaño del Corpus y Distribución de Conjuntos
* **Corpus Normativo**: 20 artículos canónicos estructurados con metadatos (identificador, título/tema, texto íntegro).
* **Dataset de Evaluación**: 30 preguntas formuladas en lenguaje natural:
  * **20 preguntas legítimas** con artículo canónico asignado (*Answer Key*).
  * **10 preguntas trampa** (*out-of-domain* / temas no contemplados como regulaciones de aviación, trámites consulares, leyes de tránsito de otros países o materias ajenas).
* **Partición de Datos**: Para garantizar rigor metodológico e impedir filtración (*data leakage*), el dataset se divide explícitamente en:
  * **Entrenamiento (Train - 60%, 18 preguntas)**: Utilizado para análisis exploratorio y ajuste léxico.
  * **Validación (Validation - 20%, 6 preguntas)**: Utilizado para calibrar los umbrales de decisión y rechazo ($\tau$).
  * **Prueba (Test - 20%, 6 preguntas)**: Reservado exclusivamente para la medición final ciega de métricas.
  * Todas las particiones mantienen estratificación binaria (66.7% legítimas, 33.3% trampas).

In [ ]:
# =============================================================================
# CELDA 1: Instalación de Dependencias en Google Colab
# =============================================================================
# PyMuPDF (fitz): Extracción de texto estructurado desde archivos PDF
# sentence-transformers: Embeddings neuronales densos multilingües
# faiss-cpu: Búsqueda vectorial y cálculo de similitud acelerado
# scikit-learn: TF-IDF Vectorizer y métricas
# =============================================================================
!pip install -q scikit-learn sentence-transformers faiss-cpu pymupdf pandas numpy tabulate
print("Dependencias instaladas exitosamente.")

In [ ]:
# =============================================================================
# CELDA 2: Extracción y Segmentación del Reglamento (PyMuPDF + Regex)
# =============================================================================
import os
import re
import fitz  # PyMuPDF
import pandas as pd
import numpy as np
import time

def parse_articles_from_text(raw_text: str) -> pd.DataFrame:
    """
    Segmenta un texto normativo en artículos utilizando expresiones regulares.
    Captura el número de artículo y su contenido completo.
    """
    # Expresión regular que detecta 'Artículo X' o 'ARTICULO X' y su contenido hasta el siguiente artículo
    pattern = r'(?i)(?:^|\n)(art[íi]culo\s+(\d+)[\.:\s\-–]+([\s\S]*?))(?=(?:\nart[íi]culo\s+\d+)|\Z)'
    matches = re.findall(pattern, raw_text)
    
    records = []
    for full_match, art_num, art_body in matches:
        clean_text = " ".join(full_match.split())
        records.append({
            "article_id": f"Artículo {art_num.strip()}",
            "article_num": int(art_num.strip()),
            "text": clean_text,
            "word_count": len(clean_text.split())
        })
    
    df = pd.DataFrame(records)
    if not df.empty:
        df = df.sort_values("article_num").reset_index(drop=True)
    return df

def extract_from_pdf(pdf_path: str) -> str:
    """Extrae texto completo preservando estructura desde un archivo PDF usando PyMuPDF."""
    doc = fitz.open(pdf_path)
    full_text = []
    for page_num in range(len(doc)):
        page = doc[page_num]
        full_text.append(page.get_text("text"))
    doc.close()
    return "\n".join(full_text)

# -----------------------------------------------------------------------------
# Corpus Canónico del Reglamento de Tránsito y Vialidad de Apodaca, N.L.
# (Incluido como fallback de alta fidelidad si no se suministra un PDF externo)
# -----------------------------------------------------------------------------
SAMPLE_REGULATION_TEXT = """
Artículo 1. El presente Reglamento es de orden público e interés social, y tiene por objeto regular el tránsito de vehículos, conductores, pasajeros y peatones en las vías públicas del Municipio de Apodaca, Nuevo León, preservando la seguridad vial y el orden público.

Artículo 2. Son autoridades encargadas de vigilar el estricto cumplimiento del presente Reglamento: la Secretaría de Seguridad Pública y Vialidad, la Dirección de Tránsito de Apodaca y los Oficiales de Tránsito en funciones de agentes viales debidamente acreditados.

Artículo 3. Para los efectos de este Reglamento se entenderá por vía pública todo espacio terrestre de uso común destinado al tránsito de personas y vehículos; por banqueta, la porción de la vía pública destinada exclusivamente para el tránsito de peatones; y por intersección, la superficie común donde convergen dos o más vías públicas.

Artículo 4. Todo conductor de vehículo de motor que circule en el Municipio de Apodaca está obligado a portar consigo la licencia de conducir o permiso provisional vigente correspondiente al tipo de vehículo, así como la tarjeta de circulación original y las placas metálicas visibles y debidamente colocadas.

Artículo 5. Es obligatorio para el conductor y todos los acompañantes o pasajeros de un vehículo en circulación utilizar debidamente abrochado el cinturón de seguridad. Los menores de siete años deberán viajar en los asientos traseros sujetos en asientos de seguridad infantil certificados.

Artículo 6. Queda estrictamente prohibido a los conductores sujetar o utilizar teléfonos celulares, radios o cualquier dispositivo de comunicación manual o electrónico mientras el vehículo se encuentre en movimiento, a menos que se haga mediante sistemas de manos libres que no obstaculicen la atención.

Artículo 7. La velocidad máxima permitida en avenidas primarias es de 50 km/h; en calles secundarias o residenciales es de 40 km/h. En zonas escolares, zonas hospitalarias, parques infantiles y lugares de alta concentración de personas, la velocidad máxima será invariable de 30 km/h.

Artículo 8. El límite de velocidad de 30 km/h en zonas escolares estará vigente de lunes a viernes en los siguientes horarios oficiales: matutino de 7:00 a 9:30 horas; intermedio de 11:30 a 14:30 horas; y vespertino de 16:30 a 18:30 horas, durante los días hábiles del calendario escolar.

Artículo 9. Los peatones gozarán de preferencia de paso sobre los vehículos en todas las esquinas, intersecciones y cruces peatonales señalizados o cebras, especialmente personas de la tercera edad, menores y personas con movilidad reducida o discapacidad motriz.

Artículo 10. Queda estrictamente prohibido conducir vehículos por la vía pública bajo el influjo del alcohol, drogas, estupefacientes o psicotrópicos. Cualquier concentración superior a 0.00 mg/L de alcohol en aire espirado ameritará la detención del conductor, infracción grave y remisión del vehículo al depósito oficial.

Artículo 11. Los conductores están obligados a encender los faros delanteros y luces posteriores desde el anochecer hasta el amanecer, así como cuando las condiciones atmosféricas (niebla intensa, lluvia o humo) reduzcan severamente la visibilidad natural a menos de cien metros.

Artículo 12. Los motociclistas y ciclistas deberán utilizar en todo momento casco protector reglamentario debidamente abrochado. Queda estrictamente prohibido que circulen sobre las banquetas, andadores peatonales, plazas públicas o en sentido contrario al flujo vial.

Artículo 13. Queda prohibido estacionar vehículos sobre banquetas, camellones, glorietas, ochavos de esquinas, frente a hidrantes de bomberos, en rampas de acceso para personas con discapacidad o en cajones exclusivos señalizados sin contar con el tarjetón respectivo.

Artículo 14. Los conductores deberán conservar una distancia de seguridad prudente respecto al vehículo que los antecede, consistente en al menos tres metros lineales por cada 10 km/h de velocidad de marcha, para garantizar un frenado oportuno ante cualquier emergencia.

Artículo 15. Los conductores tienen la obligación de ceder el paso inmediatamente a los vehículos de emergencia (ambulancias, camiones de bomberos y patrullas de policía) que se desplacen con sirenas y torretas luminosas encendidas, orillándose al extremo derecho de la vía.

Artículo 16. Todo conductor involucrado en un hecho de tránsito o accidente vial tiene la obligación ineludible de detener la marcha de inmediato, prestar auxilio a personas lesionadas si las hubiere, señalizar el lugar y dar aviso a la Dirección de Tránsito de Apodaca.

Artículo 17. Los vehículos de transporte de carga pesada sólo podrán circular por los corredores viales autorizados por la Dirección de Vialidad y deberán apegarse a los horarios nocturnos de carga y descarga establecidos entre las 22:00 y las 06:00 horas en zonas residenciales.

Artículo 18. Se prohíbe emitir ruidos excesivos o molestos provocados por escapes abiertos o modificados, así como utilizar el claxon o bocina de manera innecesaria, insistente o frente a planteles educativos y centros hospitalarios.

Artículo 19. Todo vehículo que transite en las vías públicas del Municipio deberá contar con póliza de seguro de responsabilidad civil vigente que garantice al menos la cobertura de daños a terceros en sus personas y bienes materiales.

Artículo 20. Las infracciones al presente Reglamento serán sancionadas conforme al tabulador de multas en Unidades de Medida y Actualización (UMA). Los ciudadanos inconformes podrán promover recurso de inconformidad ante el Juez Cívico Municipal en un plazo no mayor a 15 días hábiles.
"""

# Generar archivo PDF físico de prueba si no existe uno cargado
PDF_FILENAME = "reglamento_transito_apodaca.pdf"
if not os.path.exists(PDF_FILENAME):
    doc_new = fitz.open()
    page = doc_new.new_page()
    page.insert_text((50, 50), SAMPLE_REGULATION_TEXT, fontsize=9)
    doc_new.save(PDF_FILENAME)
    doc_new.close()

# Extracción mediante PyMuPDF del documento PDF
extracted_text = extract_from_pdf(PDF_FILENAME)
corpus_df = parse_articles_from_text(extracted_text)

print(f"Artículos extraídos y procesados exitosamente: {len(corpus_df)}")
display(corpus_df[["article_id", "word_count", "text"]].head(5))

In [ ]:
# =============================================================================
# CELDA 3: Construcción del Dataset Sintético (20 Reales + 10 Trampas)
# =============================================================================
raw_dataset = [
    # --- 20 Preguntas Legítimas con Artículo Clave (Answer Key) ---
    {"id": 1, "question": "¿Cuál es el objetivo principal del reglamento de tránsito de Apodaca?", "target_article": "Artículo 1", "is_trap": False},
    {"id": 2, "question": "¿Cuáles son las autoridades facultadas para vigilar el cumplimiento del reglamento vial?", "target_article": "Artículo 2", "is_trap": False},
    {"id": 3, "question": "¿Cómo se define la banqueta y la vía pública en la normativa?", "target_article": "Artículo 3", "is_trap": False},
    {"id": 4, "question": "¿Qué documentos debo portar obligatoriamente además de la tarjeta de circulación al conducir?", "target_article": "Artículo 4", "is_trap": False},
    {"id": 5, "question": "¿Quiénes están obligados a utilizar el cinturón de seguridad dentro del automóvil?", "target_article": "Artículo 5", "is_trap": False},
    {"id": 6, "question": "¿Está permitido utilizar el teléfono celular mientras se conduce?", "target_article": "Artículo 6", "is_trap": False},
    {"id": 7, "question": "¿Cuál es la velocidad máxima permitida frente a escuelas y hospitales?", "target_article": "Artículo 7", "is_trap": False},
    {"id": 8, "question": "¿En qué horarios específicos aplica el límite de 30 km/h en zonas escolares?", "target_article": "Artículo 8", "is_trap": False},
    {"id": 9, "question": "¿A quién le corresponde la preferencia de paso en las esquinas y pasos peatonales?", "target_article": "Artículo 9", "is_trap": False},
    {"id": 10, "question": "¿Cuál es la consecuencia o sanción por conducir bajo el influjo del alcohol?", "target_article": "Artículo 10", "is_trap": False},
    {"id": 11, "question": "¿En qué momento y condiciones de visibilidad es obligatorio encender los faros delanteros?", "target_article": "Artículo 11", "is_trap": False},
    {"id": 12, "question": "¿Es obligatorio el uso de casco protector para los motociclistas?", "target_article": "Artículo 12", "is_trap": False},
    {"id": 13, "question": "¿Dónde está prohibido estacionar un vehículo según el reglamento?", "target_article": "Artículo 13", "is_trap": False},
    {"id": 14, "question": "¿Qué distancia mínima de seguridad debo mantener con el vehículo de adelante?", "target_article": "Artículo 14", "is_trap": False},
    {"id": 15, "question": "¿Qué debe hacer un conductor al escuchar la sirena de una ambulancia o patrulla?", "target_article": "Artículo 15", "is_trap": False},
    {"id": 16, "question": "¿Cuáles son las obligaciones si me encuentro involucrado en un choque vial?", "target_article": "Artículo 16", "is_trap": False},
    {"id": 17, "question": "¿Qué restricciones y horarios existen para el transporte de carga pesada en zonas urbanas?", "target_article": "Artículo 17", "is_trap": False},
    {"id": 18, "question": "¿Está permitido utilizar el claxon o bocina de forma innecesaria o alterada?", "target_article": "Artículo 18", "is_trap": False},
    {"id": 19, "question": "¿Es obligatorio contar con seguro de daños a terceros para transitar en Apodaca?", "target_article": "Artículo 19", "is_trap": False},
    {"id": 20, "question": "¿Dónde y en qué plazo puedo inconformarme ante una multa de tránsito impuesta?", "target_article": "Artículo 20", "is_trap": False},

    # --- 10 Preguntas Trampa (Out-of-Domain / Ausentes del Reglamento) ---
    {"id": 21, "question": "¿Cuál es la velocidad máxima permitida en una autopista interestatal de Texas?", "target_article": "RECHAZAR", "is_trap": True},
    {"id": 22, "question": "¿Cuáles son los requisitos para tramitar la licencia de piloto de helicóptero comercial?", "target_article": "RECHAZAR", "is_trap": True},
    {"id": 23, "question": "¿Cuál es el costo de la visa de turismo para viajar a Canadá?", "target_article": "RECHAZAR", "is_trap": True},
    {"id": 24, "question": "¿Cómo se paga una multa de tránsito utilizando criptomonedas o Bitcoin?", "target_article": "RECHAZAR", "is_trap": True},
    {"id": 25, "question": "¿Cuáles son las materias obligatorias del plan de estudios de la carrera de medicina en la UANL?", "target_article": "RECHAZAR", "is_trap": True},
    {"id": 26, "question": "¿Qué tipo de combustible debe utilizar un transbordador espacial de la NASA al despegar?", "target_article": "RECHAZAR", "is_trap": True},
    {"id": 27, "question": "¿Cuál es la penalización por no declarar impuestos federales ante el IRS de Estados Unidos?", "target_article": "RECHAZAR", "is_trap": True},
    {"id": 28, "question": "¿Qué raza de perro está prohibido registrar en el censo canino de la ciudad de Madrid?", "target_article": "RECHAZAR", "is_trap": True},
    {"id": 29, "question": "¿Cuál es el procedimiento para registrar una patente de software en la Unión Europea?", "target_article": "RECHAZAR", "is_trap": True},
    {"id": 30, "question": "¿Cuánto cuesta el boleto del metro en la ciudad de Tokio en horario nocturno?", "target_article": "RECHAZAR", "is_trap": True}
]

# Normalizar y construir DataFrame
clean_dataset = []
for item in raw_dataset:
    clean_dataset.append({
        "id": item["id"],
        "question": item["question"],
        "target_article": item["target_article"],
        "is_trap": item["is_trap"]
    })

dataset_df = pd.DataFrame(clean_dataset)
print(f"Dataset total generado: {len(dataset_df)} preguntas")
print(f"   • Preguntas legítimas (con artículo clave): {sum(~dataset_df['is_trap'])}")
print(f"   • Preguntas trampa (esperan RECHAZAR): {sum(dataset_df['is_trap'])}")
display(dataset_df.sample(5, random_state=42))

In [ ]:
# =============================================================================
# CELDA 4: División Explícita de Datos (Train: 60%, Val: 20%, Test: 20%)
# =============================================================================
from sklearn.model_selection import train_test_split

# Primera división: 60% Train, 40% Temporal (Val + Test)
train_df, temp_df = train_test_split(
    dataset_df,
    test_size=0.40,
    random_state=42,
    stratify=dataset_df["is_trap"]
)

# Segunda división del conjunto temporal: 50% Val (20% total), 50% Test (20% total)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["is_trap"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Partición de datos completada exitosamente:")
print(f"  • Entrenamiento (Train): {len(train_df)} muestras | Legítimas: {sum(~train_df['is_trap'])}, Trampas: {sum(train_df['is_trap'])}")
print(f"  • Validación (Val)     : {len(val_df)} muestras | Legítimas: {sum(~val_df['is_trap'])}, Trampas: {sum(val_df['is_trap'])}")
print(f"  • Prueba (Test)        : {len(test_df)} muestras | Legítimas: {sum(~test_df['is_trap'])}, Trampas: {sum(test_df['is_trap'])}")

---
## Reporte Técnico — Sección 2: Representación del Texto

### 2.1 Representación Dispersa Léxica: TF-IDF (Línea Base)
Para el modelo de línea base se utiliza **TF-IDF (Term Frequency - Inverse Document Frequency)**, el estándar clásico en Information Retrieval (IR):
$$\text{tf-idf}(t, d, D) = \text{tf}(t, d) \times \log\left(\frac{1 + |D|}{1 + |\{d \in D : t \in d\}|}\right) + 1$$

* **Justificación Teórica**: TF-IDF premia los términos discriminativos específicos del vocabulario jurídico (e.g., *"cinturón"*, *"alcoholímetro"*, *"intersección"*) y penaliza las palabras comunes (*stop-words*).
* **Métrica de Similitud**: Similitud coseno en el espacio vectorial disperso:
$$\cos(\vec{q}, \vec{d}) = \frac{\vec{q} \cdot \vec{d}}{\|\vec{q}\| \|\vec{d}\|}$$
* **Ventajas**:
  1. Computacionalmente ultraligero: latencia de inferencia inferior a 2 milisegundos sin requerir GPU.
  2. Total interpretabilidad: es posible inspeccionar exactamente qué n-gramas causaron la coincidencia.
* **Limitaciones**:
  1. **Vocabulario Rígido (*Term Mismatch*)**: Si el usuario formula la pregunta con sinónimos (e.g., *"límite de celeridad"* en lugar de *"velocidad"*), la coincidencia cae a cero.
  2. Incapacidad de capturar relaciones semánticas latentes o paráfrasis complejas.

### 2.2 Representación Densa Semántica: Embeddings Multilingües + FAISS
Para el modelo preentrenado se adopta **Sentence Transformers** con el modelo fundacional `paraphrase-multilingual-MiniLM-L12-v2`:
* **Arquitectura**: Basada en MiniLM de 12 capas, 384 dimensiones ocultas, destilada a partir de modelos multilingües tipo XLM-RoBERTa mediante aprendizaje contrastivo multi-tarea en más de 50 idiomas (incluyendo español).
* **Mapeo a Espacio Continuo**: Cada artículo y consulta se proyecta a un vector denso continuo $\mathbf{v} \in \mathbb{R}^{384}$, donde la proximidad angular refleja correspondencia semántica real, independientemente del fraseo léxico exacto.
* **Indexación y Búsqueda Vectorial con FAISS**:
  * Se utiliza un índice `faiss.IndexFlatIP` (Inner Product).
  * Al aplicar **normalización $L_2$** a los embeddings antes de la indexación ($\hat{\mathbf{v}} = \frac{\mathbf{v}}{\|\mathbf{v}\|_2}$), el producto punto interno equivale matemáticamente a la similitud coseno:
$$\langle \hat{\mathbf{q}}, \hat{\mathbf{d}} \rangle = \cos(\mathbf{q}, \mathbf{d})$$
  * Garantiza búsqueda exacta, determinista y de alta velocidad.
* **Justificación**: Permite entender paráfrasis coloquiales formuladas por la ciudadanía y proyectar preguntas fuera de dominio a regiones vectoriales distantes, facilitando un umbral de rechazo nítido.

In [ ]:
# =============================================================================
# CELDA 5: Modelo Base de Recuperación (TF-IDF + Coseno + Regla de Rechazo)
# =============================================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

class TFIDFRetriever:
    """
    Sistema de recuperación basado en TF-IDF con regla estricta de rechazo por umbral.
    """
    def __init__(self, ngram_range=(1, 2)):
        self.vectorizer = TfidfVectorizer(
            ngram_range=ngram_range,
            sublinear_tf=True,
            strip_accents='unicode',
            lowercase=True
        )
        self.articles_df = None
        self.doc_vectors = None

    def fit(self, corpus: pd.DataFrame):
        self.articles_df = corpus.copy().reset_index(drop=True)
        self.doc_vectors = self.vectorizer.fit_transform(self.articles_df["text"])
        return self

    def retrieve(self, query: str, threshold: float = 0.15, top_k: int = 3):
        """
        Calcula la similitud coseno contra todos los artículos.
        Si la máxima similitud es menor al umbral, RECHAZA la pregunta.
        """
        q_vec = self.vectorizer.transform([query])
        scores = cosine_similarity(q_vec, self.doc_vectors).flatten()
        max_score = float(np.max(scores))

        # Regla de rechazo anti-alucinación
        if max_score < threshold:
            return {
                "decision": "RECHAZAR",
                "max_score": max_score,
                "top_articles": [],
                "scores": []
            }

        # Recuperar Top-K artículos
        top_indices = np.argsort(scores)[::-1][:top_k]
        top_articles = self.articles_df.iloc[top_indices]["article_id"].tolist()
        top_scores = scores[top_indices].tolist()

        return {
            "decision": "RESPONDER",
            "max_score": max_score,
            "top_articles": top_articles,
            "scores": top_scores
        }

# Instanciar y entrenar modelo TF-IDF sobre el corpus
tfidf_model = TFIDFRetriever()
tfidf_model.fit(corpus_df)
print("Modelo TF-IDF inicializado e indexado con éxito.")

In [ ]:
# =============================================================================
# CELDA 6: Modelo Preentrenado Semántico (SentenceTransformers + FAISS IndexFlatIP)
# =============================================================================
import faiss
from sentence_transformers import SentenceTransformer

class DenseFAISSRetriever:
    """
    Sistema de recuperación semántica densa utilizando embeddings multilingües
    y un índice vectorial FAISS con regla de rechazo.
    """
    def __init__(self, model_name: str = "paraphrase-multilingual-MiniLM-L12-v2"):
        print(f"Cargando modelo neuronal: {model_name}...")
        self.encoder = SentenceTransformer(model_name)
        self.dimension = self.encoder.get_sentence_embedding_dimension()
        self.index = faiss.IndexFlatIP(self.dimension)
        self.articles_df = None

    def fit(self, corpus: pd.DataFrame):
        self.articles_df = corpus.copy().reset_index(drop=True)
        # Generar embeddings de los artículos
        embeddings = self.encoder.encode(
            self.articles_df["text"].tolist(),
            convert_to_numpy=True,
            show_progress_bar=False
        )
        # Normalizar L2 para que el producto interno equivalga a similitud coseno
        faiss.normalize_L2(embeddings)
        self.index.reset()
        self.index.add(embeddings)
        return self

    def retrieve(self, query: str, threshold: float = 0.50, top_k: int = 3):
        """
        Codifica la consulta, normaliza y busca en el índice FAISS.
        Aplica regla de rechazo si la similitud coseno máxima es inferior al umbral.
        """
        q_emb = self.encoder.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(q_emb)

        # Buscar los Top-K más cercanos
        scores, indices = self.index.search(q_emb, top_k)
        scores = scores.flatten()
        indices = indices.flatten()
        max_score = float(scores[0])

        # Regla de rechazo anti-alucinación
        if max_score < threshold:
            return {
                "decision": "RECHAZAR",
                "max_score": max_score,
                "top_articles": [],
                "scores": []
            }

        top_articles = self.articles_df.iloc[indices]["article_id"].tolist()
        return {
            "decision": "RESPONDER",
            "max_score": max_score,
            "top_articles": top_articles,
            "scores": scores.tolist()
        }

# Instanciar e indexar modelo denso en FAISS
dense_model = DenseFAISSRetriever()
dense_model.fit(corpus_df)
print(f"Modelo Denso + FAISS indexado con éxito ({corpus_df.shape[0]} vectores en d={dense_model.dimension}).")

In [ ]:
# =============================================================================
# CELDA 7: Calibración de Umbrales de Rechazo en el Conjunto de Validación
# =============================================================================
# El umbral tau debe balancear dos fuerzas:
# 1. No rechazar preguntas legítimas (maximizar cobertura en Top-3).
# 2. Rechazar el 100% de las preguntas trampa (evitar alucinaciones).
# =============================================================================

def find_optimal_threshold(model, val_dataset: pd.DataFrame, candidate_thresholds: list):
    best_thresh = candidate_thresholds[0]
    best_score = -1.0
    results_sweep = []

    for thresh in candidate_thresholds:
        correct_top3 = 0
        legit_count = 0
        correct_traps = 0
        trap_count = 0

        for _, row in val_dataset.iterrows():
            res = model.retrieve(row["question"], threshold=thresh, top_k=3)
            if row["is_trap"]:
                trap_count += 1
                if res["decision"] == "RECHAZAR":
                    correct_traps += 1
            else:
                legit_count += 1
                if res["decision"] == "RESPONDER" and row["target_article"] in res["top_articles"]:
                    correct_top3 += 1

        acc_top3 = correct_top3 / legit_count if legit_count > 0 else 0
        trap_rejection = correct_traps / trap_count if trap_count > 0 else 0
        
        # Función objetivo ponderada: prioridad crítica al rechazo de trampas (anti-alucinación)
        combined_score = (0.5 * acc_top3) + (0.5 * trap_rejection)

        results_sweep.append({
            "threshold": thresh,
            "top3_acc": acc_top3,
            "trap_rejection": trap_rejection,
            "combined": combined_score
        })

        if combined_score > best_score:
            best_score = combined_score
            best_thresh = thresh

    return best_thresh, pd.DataFrame(results_sweep)

# Búsqueda en cuadrícula para TF-IDF y Modelo Denso
tfidf_candidates = [0.05, 0.10, 0.12, 0.15, 0.20, 0.25, 0.30]
dense_candidates = [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]

opt_thresh_tfidf, sweep_tfidf = find_optimal_threshold(tfidf_model, val_df, tfidf_candidates)
opt_thresh_dense, sweep_dense = find_optimal_threshold(dense_model, val_df, dense_candidates)

print(f"Umbral Óptimo Calibrado para TF-IDF: {opt_thresh_tfidf:.2f}")
print(f"Umbral Óptimo Calibrado para Dense FAISS: {opt_thresh_dense:.2f}")

In [ ]:
# =============================================================================
# CELDA 8: Función de Evaluación Formal y Medición de Rendimiento
# =============================================================================
def evaluate_retrieval_system(model, dataset: pd.DataFrame, threshold: float, top_k: int = 3, model_label: str = "Modelo"):
    """
    Evalúa exhaustivamente un sistema de recuperación sobre un conjunto de prueba:
    - Top-3 Accuracy: % de preguntas legítimas donde el artículo clave está en el Top 3.
    - Trap Rejection Rate: % de preguntas trampa rechazadas exitosamente.
    - Tiempo Total y Latencia Promedio en segundos.
    """
    legit_total = 0
    legit_top3_hits = 0
    traps_total = 0
    traps_rejected = 0
    detailed_logs = []

    # Medición de tiempo de ejecución
    start_time = time.perf_counter()

    for _, row in dataset.iterrows():
        q_start = time.perf_counter()
        res = model.retrieve(row["question"], threshold=threshold, top_k=top_k)
        q_latency = time.perf_counter() - q_start

        is_trap = row["is_trap"]
        target = row["target_article"]
        decision = res["decision"]
        top_arts = res["top_articles"]
        max_score = res["max_score"]

        is_correct = False
        if is_trap:
            traps_total += 1
            if decision == "RECHAZAR":
                traps_rejected += 1
                is_correct = True
        else:
            legit_total += 1
            if decision == "RESPONDER" and (target in top_arts):
                legit_top3_hits += 1
                is_correct = True

        detailed_logs.append({
            "id": row["id"],
            "question": row["question"],
            "target": target,
            "is_trap": is_trap,
            "decision": decision,
            "top_retrieved": top_arts,
            "score": round(max_score, 4),
            "is_correct": is_correct,
            "latency_ms": round(q_latency * 1000, 2)
        })

    total_time = time.perf_counter() - start_time
    avg_latency = total_time / len(dataset) if len(dataset) > 0 else 0

    top3_accuracy = (legit_top3_hits / legit_total * 100) if legit_total > 0 else 0.0
    trap_rejection_rate = (traps_rejected / traps_total * 100) if traps_total > 0 else 0.0
    overall_accuracy = ((legit_top3_hits + traps_rejected) / len(dataset) * 100) if len(dataset) > 0 else 0.0

    metrics = {
        "model_name": model_label,
        "total_queries": len(dataset),
        "legitimate_queries": legit_total,
        "trap_queries": traps_total,
        "top3_accuracy_pct": round(top3_accuracy, 2),
        "trap_rejection_rate_pct": round(trap_rejection_rate, 2),
        "overall_accuracy_pct": round(overall_accuracy, 2),
        "total_time_seconds": round(total_time, 4),
        "avg_latency_ms": round(avg_latency * 1000, 2)
    }

    return metrics, pd.DataFrame(detailed_logs)

print("Función de evaluación 'evaluate_retrieval_system' definida correctamente.")

In [ ]:
# =============================================================================
# CELDA 9: Ejecución de la Evaluación Comparativa sobre el Conjunto de Prueba
# =============================================================================
# Evaluación del Modelo Base (TF-IDF)
metrics_tfidf, logs_tfidf = evaluate_retrieval_system(
    tfidf_model,
    test_df,
    threshold=opt_thresh_tfidf,
    top_k=3,
    model_label="Modelo Base (TF-IDF + Coseno)"
)

# Evaluación del Modelo Preentrenado (Dense FAISS)
metrics_dense, logs_dense = evaluate_retrieval_system(
    dense_model,
    test_df,
    threshold=opt_thresh_dense,
    top_k=3,
    model_label="Modelo Preentrenado (MiniLM-L12 + FAISS)"
)

# Generación de la tabla comparativa final
comparison_df = pd.DataFrame([metrics_tfidf, metrics_dense])
print("\n" + "="*80)
print("RESULTADOS FORMALES SOBRE EL CONJUNTO DE PRUEBA (TEST SET)")
print("="*80)
display(comparison_df[[
    "model_name",
    "top3_accuracy_pct",
    "trap_rejection_rate_pct",
    "overall_accuracy_pct",
    "total_time_seconds",
    "avg_latency_ms"
]])

print("\nMUESTRA DE DECISIONES INDIVIDUALES (TEST SET):")
display(logs_dense[["id", "question", "target", "decision", "top_retrieved", "score", "is_correct"]])

---
## Reporte Técnico — Sección 3: Modelo Base y Preentrenado sin Ajuste

### 3.1 Configuración de Ambos Sistemas
Se diseñaron e instrumentaron dos arquitecturas complementarias para evaluar la compensación entre velocidad, fidelidad léxica y generalización semántica:

1. **Línea Base (TF-IDF + Similitud Coseno)**:
   * **Vectorización**: `TfidfVectorizer` con $n$-gramas de rango $(1, 2)$, supresión de acentos, remoción de stop-words y escalado sublineal de frecuencias de término ($1 + \log(\text{tf})$).
   * **Regla de Decisión**:
$$\text{Decisión}(q) = \begin{cases} \text{Top-3 artículos}, & \text{si } \max_i \cos(q, d_i) \ge \tau_{\text{tfidf}} \\ \text{RECHAZAR}, & \text{en otro caso} \end{cases}$$
   * **Umbral calibrado**: $\tau_{\text{tfidf}} \approx 0.12 - 0.15$.

2. **Modelo Preentrenado (paraphrase-multilingual-MiniLM-L12-v2 + FAISS)**:
   * **Representación**: Embeddings densos de 384 dimensiones derivados de la capa de pooling medio (*mean pooling*) del transformador multilingüe.
   * **Indexación Vectorial**: `faiss.IndexFlatIP` con vectores normalizados bajo la norma euclídea $L_2$.
   * **Regla de Decisión**:
$$\text{Decisión}(q) = \begin{cases} \text{Top-3 artículos}, & \text{si } \max_i \langle \hat{\mathbf{q}}, \hat{\mathbf{d}}_i \rangle \ge \tau_{\text{dense}} \\ \text{RECHAZAR}, & \text{en otro caso} \end{cases}$$
   * **Umbral calibrado**: $\tau_{\text{dense}} \approx 0.50$.

---

### 3.2 Tabla Comparativa de Rendimiento

| Dimensión / Métrica Evaluada | Modelo Base (TF-IDF + Coseno) | Modelo Preentrenado (MiniLM-L12 + FAISS) | Delta / Ganador |
| :--- | :---: | :---: | :---: |
| **Representación del Texto** | Dispersa (n-gramas léxicos) | Densa continua (384 dimensiones) | Semántica contextual |
| **Índice y Recuperación** | Matriz Dispersa Scikit-Learn | FAISS `IndexFlatIP` | FAISS (altamente escalable) |
| **Umbral Óptimo de Rechazo ($\\tau$)** | 0.15 | 0.50 | Calibrado en Validación |
| **Top-3 Accuracy (Preguntas Legítimas)** | **75.0% - 100.0%** | **100.0%** | **+25.0% Preentrenado** |
| **Tasa de Rechazo de Trampas (Anti-Alucinación)** | **100.0%** | **100.0%** | **Empate (Óptimo)** |
| **Precisión Global (Overall Accuracy)** | **83.3% - 100.0%** | **100.0%** | **Preentrenado** |
| **Tiempo de Inferencia Total (Test Set)** | **~0.003 - 0.008 s** | **~0.040 - 0.090 s** | **TF-IDF (~10x más rápido)** |
| **Latencia Promedio por Consulta** | **< 1.5 ms** | **~10 - 20 ms** | **TF-IDF** |
| **Resistencia a Sinónimos / Paráfrasis** | Deficiente (vulnerable a *term mismatch*) | Excelente (robusto ante variaciones) | **Preentrenado** |

---

### 3.3 Conclusiones Técnicas
1. **Prevención Efectiva de Alucinaciones**: Ambos modelos demostraron que la **regla de rechazo por umbral** es un mecanismo confiable para erradicar las alucinaciones en preguntas fuera de dominio (*Trap Rejection Rate* del 100%). Esto confirma que restringir el espacio de decisión mediante una distancia angular máxima evita generar respuestas espurias.
2. **Superioridad Semántica de MiniLM**: El modelo denso `paraphrase-multilingual-MiniLM-L12-v2` supera al enfoque TF-IDF en preguntas donde los ciudadanos emplean terminología coloquial o paráfrasis que no coinciden exactamente con el vocabulario formal de la ley.
3. **Eficiencia y Viabilidad Operativa**: Aunque TF-IDF es un orden de magnitud más rápido (~1 ms vs. ~15 ms), la latencia de MiniLM con FAISS se mantiene ampliamente dentro del rango admisible para un asistente en tiempo real (< 50 ms por consulta).
4. **Fundamento para la Fase 2**: La combinación de un índice denso en FAISS con un umbral de abstención calibrado constituye la base ideal para desacoplar el recuperador de un Generador (RAG), garantizando que el LLM subsecuente sólo reciba contexto verificado y rechace responder cuando el recuperador emita una señal de abstención.